# 03 - Preprocesamiento y Feature Engineering
Limpieza, transformaciones y construcción del pipeline reproducible.

In [1]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import joblib

df = pd.read_csv('../data/raw/smoking_labeled.csv')
print(f"Dataset cargado: {df.shape}")
df.head(3)


Dataset cargado: (50000, 27)


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking
0,0,F,40,155,60,3.38,0.04,0.04,0.04,0.04,...,0.51,0.04,0.00,0.75,0.79,1.13,Y,0,Y,0
1,1,F,40,160,60,3.38,0.01,0.00,0.04,0.04,...,0.50,0.04,0.00,0.92,0.79,0.75,Y,0,Y,0
2,2,M,55,170,60,3.33,0.01,0.01,0.04,0.04,...,0.63,0.04,0.04,0.88,0.67,0.92,Y,0,N,1


In [2]:

# ── 1. Análisis de nulos ───────────────────────────────────────────────────
print("Nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal filas con al menos un nulo: {df.isnull().any(axis=1).sum()}")


Nulos por columna:
Series([], dtype: int64)

Total filas con al menos un nulo: 0


In [3]:

# ── 2. Separar features y target ──────────────────────────────────────────
TARGET = 'smoking'
COLS_TO_DROP = []  # agregar columnas irrelevantes si las hay (ej: IDs)

X = df.drop(columns=[TARGET] + COLS_TO_DROP)
y = df[TARGET]

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")


X shape: (50000, 26)
y distribution:
smoking
0    31671
1    18329
Name: count, dtype: int64


In [4]:

# ── 3. Identificar tipos de columnas ──────────────────────────────────────
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Columnas numéricas ({len(num_cols)}): {num_cols}")
print(f"Columnas categóricas ({len(cat_cols)}): {cat_cols}")


Columnas numéricas (23): ['ID', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries']
Columnas categóricas (3): ['gender', 'oral', 'tartar']


In [5]:

# ── 4. Feature Engineering ────────────────────────────────────────────────
# Ejemplo de nuevas features (ajustar según columnas disponibles)
def feature_engineering(df_in):
    df_out = df_in.copy()

    # Si existen columnas de presión arterial sistólica/diastólica
    if 'systolic' in df_out.columns and 'relaxation' in df_out.columns:
        df_out['pulse_pressure'] = df_out['systolic'] - df_out['relaxation']

    # Si hay colesterol HDL y LDL
    if 'HDL' in df_out.columns and 'LDL' in df_out.columns:
        df_out['cholesterol_ratio'] = df_out['LDL'] / (df_out['HDL'] + 1e-9)

    # Si hay hemoglobina
    if 'hemoglobin' in df_out.columns:
        df_out['hemoglobin_sq'] = df_out['hemoglobin'] ** 2

    # BMI si hay height y weight
    if 'height(cm)' in df_out.columns and 'weight(kg)' in df_out.columns:
        df_out['bmi'] = df_out['weight(kg)'] / (df_out['height(cm)'] / 100) ** 2

    return df_out

X_fe = feature_engineering(X)
print(f"Features originales: {X.shape[1]}")
print(f"Features tras engineering: {X_fe.shape[1]}")


Features originales: 26
Features tras engineering: 30


In [7]:

# ── 5. Actualizar listas de columnas ──────────────────────────────────────
num_cols_fe = X_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols_fe = X_fe.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Columnas numéricas finales: {len(num_cols_fe)}")
print(f"Columnas categóricas finales: {len(cat_cols_fe)}")


Columnas numéricas finales: 27
Columnas categóricas finales: 3


In [8]:

# ── 6. Construcción del Pipeline de preprocesamiento ──────────────────────
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

if cat_cols_fe:
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, num_cols_fe),
        ('cat', categorical_transformer, cat_cols_fe)
    ])
else:
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, num_cols_fe)
    ])

print("Pipeline de preprocesamiento construido.")


Pipeline de preprocesamiento construido.


In [9]:

# ── 7. División train/test ────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Distribución: {y_train.value_counts().to_dict()}")
print(f"Test:  {X_test.shape}  | Distribución: {y_test.value_counts().to_dict()}")


Train: (40000, 30) | Distribución: {0: 25337, 1: 14663}
Test:  (10000, 30)  | Distribución: {0: 6334, 1: 3666}


In [10]:

# ── 8. Guardar datos procesados y pipeline ────────────────────────────────
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump({'num_cols': num_cols_fe, 'cat_cols': cat_cols_fe,
             'feature_engineering_fn': feature_engineering}, '../models/feature_config.pkl')

print("Datos procesados y pipeline guardados correctamente.")


Datos procesados y pipeline guardados correctamente.
